# Reproducibility and archive consistency checks

**Author: Mohammad ALbaraj**

[Notebook index](README.md) · [Main project](../README.md)

Run cells from top to bottom. Saved results and fresh examples are labeled separately. Scientific source modules remain the authoritative implementation.


In [1]:
from pathlib import Path
import sys, json, csv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Code

# Works when launched from the repository root or any notebook subdirectory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'CITATION.cff').exists() and (p / 'all work').is_dir())
if str(ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks'))
from notebook_support import (METHODS, ORIGINAL, NOMINAL, ROBUST, study, load_json,
    load_csv, show_table, show_source, source_index, load_runs, fitness_summary,
    convergence_plot, validation_rows, validation_summary, figures, file_links,
    show_figures, gallery, isolated_python, demo_source)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})
print('Project loaded. Default cells do not launch full optimization campaigns.')


Project loaded. Default cells do not launch full optimization campaigns.


## Recorded review scope
The earlier archive review checked 804 strict-budget ledgers and 46 source hashes. Nine unit suites passed; four were blocked by Windows temporary-file permissions. Those outcomes do not imply a full optimization rerun.


In [2]:
audit=load_json(ROOT/'docs/data_checks.json')
show_table(audit['groups'])
print('Previously checked ledgers:',audit['ledger_count'])


study,ledgers_checked,summary_rows_checked
all work/30 seed equal budget study,150,5
all work/robustness 30 seed study,150,5
pid using alns,84,2
pid using great deluge,84,2
pid using pure random walk,84,2
pid using shrinking ball random search,84,2
pid using threshold accepting,84,2
pid using vnd,84,2


Previously checked ledgers: 804


## Fresh source-contract hash check
This cell actually recomputes the source hashes now. A mismatch prevents treating archived runs as outputs of the modified implementation.


In [3]:
import hashlib
checks=[]
for contract_path in sorted(ROOT.rglob('results/contract.json')):
    folder=contract_path.parent.parent
    for name,expected in load_json(contract_path)['code_sha256'].items():
        actual=hashlib.sha256((folder/name).read_bytes()).hexdigest()
        checks.append({'study':folder.relative_to(ROOT).as_posix(),'file':name,'matches':actual==expected})
show_table(checks,limit=60)
assert all(c['matches'] for c in checks), 'A frozen scientific source hash changed.'


study,file,matches
all work/30 seed equal budget study,benchmark.py,True
all work/30 seed equal budget study,model.py,True
all work/30 seed equal budget study,study.py,True
all work/30 seed equal budget study,dsa_reference.py,True
all work/robustness 30 seed study,benchmark.py,True
all work/robustness 30 seed study,model.py,True
all work/robustness 30 seed study,study.py,True
all work/robustness 30 seed study,robust_protocol.py,True
all work/robustness 30 seed study,run_robust.py,True
all work/robustness 30 seed study,dsa_reference.py,True


## Fresh ledger check: one seed per method and regime
This verifies 22 ledgers by default. Set CHECK_ALL_RUNS=True to check all 660 main-run ledgers. It checks structure, unique coordinates, bounds, monotone archives, and stored best fitness—not all objective simulations.


In [4]:
CHECK_ALL_RUNS=False
checked=0
for method in METHODS:
    for mode in ['nominal','robust']:
        records=load_runs(method,mode)
        for path,record in (records if CHECK_ALL_RUNS else records[:1]):
            with np.load(path.with_name(path.stem+'_evaluations.npz'),allow_pickle=False) as data:
                ledger=data['ledger']
                assert ledger.shape==(3264,12)
                assert np.isfinite(ledger).all()
                assert len(np.unique(np.round(ledger[:,4:],13),axis=0))==3264
                assert np.all((ledger[:,4:]>=0)&(ledger[:,4:]<=1))
                np.testing.assert_allclose(ledger[:,3],np.minimum.accumulate(ledger[:,2]),rtol=0,atol=1e-13)
                assert np.isclose(record['fitness'],ledger[:,2].min(),rtol=1e-13,atol=1e-15)
            checked+=1
print('Freshly checked main-run ledgers:',checked)


Freshly checked main-run ledgers: 22


## Reproduction boundaries
Use the study-specific full-run commands only after checking the requirements and intended output folder. Notebooks never start a complete campaign automatically. Do not overwrite saved experiments with altered code, seeds, fitness definitions, or settings. Small notebook examples are teaching demonstrations, not replacements for the archived runs.
